In [14]:
import torch
print(torch.cuda.is_available())      # doit afficher True
print(torch.cuda.get_device_name(0))  # doit afficher Tesla T4

True
Tesla T4


In [15]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [16]:
import os

# Créer la structure dans ton Drive
os.makedirs('/content/drive/MyDrive/geoscanner/dataset', exist_ok=True)
os.makedirs('/content/drive/MyDrive/geoscanner/models',  exist_ok=True)

# Vérification
print("✅ Dossiers créés :")
print(os.listdir('/content/drive/MyDrive/geoscanner'))

✅ Dossiers créés :
['models', 'dataset']


In [4]:
!pip install kaggle -q

In [17]:
from google.colab import files

# Uploader ton kaggle.json
files.upload()  # sélectionne le fichier kaggle.json téléchargé

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"oussamaaitmohamed","key":"f19be642cd070b7fb270ad3fbe9d12b7"}'}

In [18]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle configuré !")

✅ Kaggle configuré !


In [19]:
!kaggle datasets download -d stealthtechnologies/rock-classification \
    -p /content/drive/MyDrive/geoscanner/dataset/

print("✅ Dataset téléchargé !")

Dataset URL: https://www.kaggle.com/datasets/stealthtechnologies/rock-classification
License(s): MIT
100% 213M/213M [00:01<00:00, 158MB/s]

✅ Dataset téléchargé !


In [ ]:
!unzip /content/drive/MyDrive/geoscanner/dataset/rock-classification.zip \
    -d /content/drive/MyDrive/geoscanner/dataset/

print("✅ Dataset décompressé !")

In [21]:
BASE = '/content/drive/MyDrive/geoscanner/dataset'

for split in ['train', 'valid', 'test']:
    path = os.path.join(BASE, split)
    if os.path.exists(path):
        classes = os.listdir(path)
        print(f"{split} → {len(classes)} classes : {classes}")

In [22]:
import os

BASE = '/content/drive/MyDrive/geoscanner/dataset'

# Voir tout ce qui est dans le dossier dataset
print("Contenu de dataset/ :")
for item in os.listdir(BASE):
    print(f"  {item}")

Contenu de dataset/ :
  Rock Data
  rock-classification.zip


In [23]:
import os

BASE = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'

for split in ['train', 'valid', 'test']:
    path = os.path.join(BASE, split)
    if os.path.exists(path):
        classes = os.listdir(path)
        total_images = sum(
            len(os.listdir(os.path.join(path, cls)))
            for cls in classes
        )
        print(f"{split:6} → {len(classes)} classes — {total_images} images")

train  → 9 classes — 3687 images
valid  → 9 classes — 351 images
test   → 9 classes — 174 images


### Partie Entrainement

In [24]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

In [3]:
BASE      = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'
MODEL_DIR = '/content/drive/MyDrive/geoscanner/models'
MODEL_PATH = os.path.join(MODEL_DIR, 'resnet50_best.pth')
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

Device : cuda


In [5]:
# Augmentation plus agressive dans train_transform
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),          # ← nouveau
    transforms.RandomRotation(30),            # ← nouveau
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2                        # ← nouveau
    ),
    transforms.RandomGrayscale(p=0.1),        # ← nouveau
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [20]:
model = models.resnet50(weights='IMAGENET1K_V1')

# Geler toutes les couches
for param in model.parameters():
    param.requires_grad = False

# Remplacer la dernière couche pour 9 classes
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 9)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()

print("✅ Modèle prêt")
print(f"Nombre de classes : 9")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]


✅ Modèle prêt
Nombre de classes : 9


##### Phase A (5 epochs)

In [22]:
model.fc.requires_grad_(True)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

EPOCHS_A = 5
best_acc = 0.0
history  = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("PHASE A — Entraînement couche FC uniquement  lr=0.001")
print("=" * 55)

for epoch in range(EPOCHS_A):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss   / len(valid_loader))
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

    print(f"Epoch {epoch+1}/{EPOCHS_A} — "
          f"Train Loss: {train_loss/len(train_loader):.4f} — "
          f"Val Loss: {val_loss/len(valid_loader):.4f} — "
          f"Val Acc: {val_acc:.2f}%")

print(f"\n✅ Meilleure accuracy Phase A : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé dans Drive")

PHASE A — Entraînement couche FC uniquement  lr=0.001
Epoch 1/5 — Train Loss: 1.6426 — Val Loss: 1.2461 — Val Acc: 60.40%
Epoch 2/5 — Train Loss: 1.2341 — Val Loss: 1.1568 — Val Acc: 60.68%
Epoch 3/5 — Train Loss: 1.1533 — Val Loss: 1.1009 — Val Acc: 62.96%
Epoch 4/5 — Train Loss: 1.0975 — Val Loss: 1.1402 — Val Acc: 59.26%
Epoch 5/5 — Train Loss: 1.0659 — Val Loss: 1.1509 — Val Acc: 62.68%

✅ Meilleure accuracy Phase A : 62.96%
✅ Modèle sauvegardé dans Drive


##### Phase B (15 epochs)

In [23]:
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.0001},
    {'params': model.layer4.parameters(), 'lr': 0.0001},
    {'params': model.fc.parameters(),     'lr': 0.0001},
])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

EPOCHS_B = 15

print("=" * 55)
print("PHASE B — Fine-tuning layer3+layer4+FC  lr=0.0001")
print("=" * 55)

for epoch in range(EPOCHS_B):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss   / len(valid_loader))
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

    print(f"Epoch {epoch+1}/{EPOCHS_B} — "
          f"Train Loss: {train_loss/len(train_loader):.4f} — "
          f"Val Loss: {val_loss/len(valid_loader):.4f} — "
          f"Val Acc: {val_acc:.2f}%")

print(f"\n✅ Meilleure accuracy finale : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

PHASE B — Fine-tuning layer3+layer4+FC  lr=0.0001
Epoch 1/15 — Train Loss: 0.7494 — Val Loss: 1.0939 — Val Acc: 70.09%
Epoch 2/15 — Train Loss: 0.2708 — Val Loss: 1.2353 — Val Acc: 68.09%
Epoch 3/15 — Train Loss: 0.1625 — Val Loss: 1.4059 — Val Acc: 69.23%
Epoch 4/15 — Train Loss: 0.1375 — Val Loss: 1.4940 — Val Acc: 71.23%
Epoch 5/15 — Train Loss: 0.1234 — Val Loss: 1.6950 — Val Acc: 68.95%
Epoch 6/15 — Train Loss: 0.0885 — Val Loss: 1.5673 — Val Acc: 70.66%
Epoch 7/15 — Train Loss: 0.1161 — Val Loss: 1.6110 — Val Acc: 69.52%
Epoch 8/15 — Train Loss: 0.0537 — Val Loss: 1.5058 — Val Acc: 69.52%
Epoch 9/15 — Train Loss: 0.0419 — Val Loss: 1.4923 — Val Acc: 70.09%
Epoch 10/15 — Train Loss: 0.0264 — Val Loss: 1.4340 — Val Acc: 73.22%
Epoch 11/15 — Train Loss: 0.0220 — Val Loss: 1.4774 — Val Acc: 69.80%
Epoch 12/15 — Train Loss: 0.0191 — Val Loss: 1.4083 — Val Acc: 72.93%
Epoch 13/15 — Train Loss: 0.0141 — Val Loss: 1.4220 — Val Acc: 72.08%
Epoch 14/15 — Train Loss: 0.0131 — Val Loss: 1.47

###  Réentraîner 1 avec ces corrections

##### Cellule 1 — Réinitialiser le modèle

In [24]:
# Repartir d'un modèle propre
model = models.resnet50(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 9)
)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
print("✅ Modèle réinitialisé")

✅ Modèle réinitialisé


#### Cellule 2 — Phase A corrigée (10 epochs)

In [27]:
model.fc.requires_grad_(True)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

EPOCHS_A = 10
best_acc = 0.0
patience_counter = 0
PATIENCE = 4  # Early stopping
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("PHASE A — FC uniquement  lr=0.001")
print("=" * 55)

for epoch in range(EPOCHS_A):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(valid_loader))
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        patience_counter += 1

    print(f"Epoch {epoch+1}/{EPOCHS_A} — "
          f"Train Loss: {train_loss/len(train_loader):.4f} — "
          f"Val Loss: {val_loss/len(valid_loader):.4f} — "
          f"Val Acc: {val_acc:.2f}% "
          f"{'⭐ meilleur' if val_acc == best_acc else ''}")

    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping à l'epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy Phase A : {best_acc:.2f}%")

PHASE A — FC uniquement  lr=0.001
Epoch 1/10 — Train Loss: 1.5931 — Val Loss: 1.3582 — Val Acc: 53.56% ⭐ meilleur
Epoch 2/10 — Train Loss: 1.2636 — Val Loss: 1.1327 — Val Acc: 60.68% ⭐ meilleur
Epoch 3/10 — Train Loss: 1.2056 — Val Loss: 1.1223 — Val Acc: 60.68% ⭐ meilleur
Epoch 4/10 — Train Loss: 1.1149 — Val Loss: 1.0855 — Val Acc: 61.54% ⭐ meilleur
Epoch 5/10 — Train Loss: 1.1228 — Val Loss: 1.0883 — Val Acc: 62.39% ⭐ meilleur
Epoch 6/10 — Train Loss: 1.0645 — Val Loss: 1.0616 — Val Acc: 65.81% ⭐ meilleur
Epoch 7/10 — Train Loss: 1.0546 — Val Loss: 1.0833 — Val Acc: 62.96% 
Epoch 8/10 — Train Loss: 1.0211 — Val Loss: 1.0714 — Val Acc: 63.82% 
Epoch 9/10 — Train Loss: 1.0006 — Val Loss: 1.0942 — Val Acc: 63.25% 
Epoch 10/10 — Train Loss: 0.9913 — Val Loss: 1.0581 — Val Acc: 63.53% 

⏹️  Early stopping à l'epoch 10

✅ Meilleure accuracy Phase A : 65.81%


#### Cellule 3 — Phase B corrigée (lr 10x plus petit)

In [30]:
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

# lr beaucoup plus petit pour éviter l'overfitting
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.00001},
    {'params': model.layer4.parameters(), 'lr': 0.00005},
    {'params': model.fc.parameters(),     'lr': 0.0001},
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

EPOCHS_B = 20
patience_counter = 0
PATIENCE = 5

print("=" * 55)
print("PHASE B — Fine-tuning layer3+layer4+FC")
print("=" * 55)

for epoch in range(EPOCHS_B):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(valid_loader))
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1}/{EPOCHS_B} — "
              f"Train Loss: {train_loss/len(train_loader):.4f} — "
              f"Val Loss: {val_loss/len(valid_loader):.4f} — "
              f"Val Acc: {val_acc:.2f}% ⭐ meilleur")
    else:
        patience_counter += 1
        print(f"Epoch {epoch+1}/{EPOCHS_B} — "
              f"Train Loss: {train_loss/len(train_loader):.4f} — "
              f"Val Loss: {val_loss/len(valid_loader):.4f} — "
              f"Val Acc: {val_acc:.2f}%")

    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping à l'epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy finale : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

PHASE B — Fine-tuning layer3+layer4+FC
Epoch 1/20 — Train Loss: 0.8042 — Val Loss: 0.9743 — Val Acc: 64.39%
Epoch 2/20 — Train Loss: 0.4555 — Val Loss: 1.0592 — Val Acc: 69.80% ⭐ meilleur
Epoch 3/20 — Train Loss: 0.2759 — Val Loss: 1.2131 — Val Acc: 68.95%
Epoch 4/20 — Train Loss: 0.1767 — Val Loss: 1.2508 — Val Acc: 69.23%
Epoch 5/20 — Train Loss: 0.1303 — Val Loss: 1.3525 — Val Acc: 69.23%
Epoch 6/20 — Train Loss: 0.1102 — Val Loss: 1.3540 — Val Acc: 70.09% ⭐ meilleur
Epoch 7/20 — Train Loss: 0.0814 — Val Loss: 1.5409 — Val Acc: 66.67%
Epoch 8/20 — Train Loss: 0.0774 — Val Loss: 1.5791 — Val Acc: 69.23%
Epoch 9/20 — Train Loss: 0.0684 — Val Loss: 1.6171 — Val Acc: 71.79% ⭐ meilleur
Epoch 10/20 — Train Loss: 0.0671 — Val Loss: 1.5773 — Val Acc: 70.94%
Epoch 11/20 — Train Loss: 0.0635 — Val Loss: 1.6323 — Val Acc: 69.52%
Epoch 12/20 — Train Loss: 0.0591 — Val Loss: 1.7890 — Val Acc: 67.52%
Epoch 13/20 — Train Loss: 0.0376 — Val Loss: 1.7568 — Val Acc: 71.23%
Epoch 14/20 — Train Loss: 0

### Réentraîner 2 avec ces corrections

##### Cellule 1 — Réinitialiser

In [31]:
model = models.resnet50(weights='IMAGENET1K_V1')

# Geler TOUT le réseau
for param in model.parameters():
    param.requires_grad = False

# FC plus robuste
model.fc = nn.Sequential(
    nn.Dropout(0.6),
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Dropout(0.4),
    nn.Linear(256, 9)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
print("✅ Modèle réinitialisé")

✅ Modèle réinitialisé


##### Cellule 2 — Entraînement FC uniquement (30 epochs)

In [32]:
# Seule la FC s'entraîne
model.fc.requires_grad_(True)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=4, factor=0.3
)

EPOCHS   = 30
best_acc = 0.0
patience_counter = 0
PATIENCE = 8
history  = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("ENTRAÎNEMENT FC — 30 epochs max  lr=0.001")
print("=" * 55)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS} — "
              f"Train Loss: {tl:.4f} — "
              f"Val Loss: {vl:.4f} — "
              f"Val Acc: {val_acc:.2f}% ⭐")
    else:
        patience_counter += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS} — "
              f"Train Loss: {tl:.4f} — "
              f"Val Loss: {vl:.4f} — "
              f"Val Acc: {val_acc:.2f}%")

    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping à l'epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

ENTRAÎNEMENT FC — 30 epochs max  lr=0.001
Epoch 01/30 — Train Loss: 1.6606 — Val Loss: 1.3780 — Val Acc: 61.25% ⭐
Epoch 02/30 — Train Loss: 1.4796 — Val Loss: 1.3533 — Val Acc: 61.82% ⭐
Epoch 03/30 — Train Loss: 1.4293 — Val Loss: 1.3403 — Val Acc: 63.25% ⭐
Epoch 04/30 — Train Loss: 1.3968 — Val Loss: 1.3444 — Val Acc: 62.11%
Epoch 05/30 — Train Loss: 1.3753 — Val Loss: 1.3241 — Val Acc: 63.53% ⭐
Epoch 06/30 — Train Loss: 1.3605 — Val Loss: 1.3272 — Val Acc: 62.39%
Epoch 07/30 — Train Loss: 1.3788 — Val Loss: 1.3190 — Val Acc: 63.25%
Epoch 08/30 — Train Loss: 1.3727 — Val Loss: 1.3090 — Val Acc: 63.53%
Epoch 09/30 — Train Loss: 1.3496 — Val Loss: 1.3606 — Val Acc: 61.82%
Epoch 10/30 — Train Loss: 1.3414 — Val Loss: 1.3160 — Val Acc: 63.53%
Epoch 11/30 — Train Loss: 1.3438 — Val Loss: 1.3108 — Val Acc: 64.10% ⭐
Epoch 12/30 — Train Loss: 1.3288 — Val Loss: 1.3216 — Val Acc: 62.96%
Epoch 13/30 — Train Loss: 1.2971 — Val Loss: 1.3235 — Val Acc: 62.68%
Epoch 14/30 — Train Loss: 1.2959 — Val

### Réentraîner 3 avec ces corrections

##### Cellule 1 — Repartir proprement

In [33]:
model = models.resnet50(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 9)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
print("✅ Modèle réinitialisé — architecture simple")

✅ Modèle réinitialisé — architecture simple


##### Cellule 2 — Phase A : FC uniquement (10 epochs)

In [34]:
model.fc.requires_grad_(True)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

EPOCHS_A     = 10
best_acc     = 0.0
patience_ctr = 0
PATIENCE     = 4
history      = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("PHASE A — FC uniquement")
print("=" * 55)

for epoch in range(EPOCHS_A):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Phase A terminée — Meilleure acc : {best_acc:.2f}%")

PHASE A — FC uniquement
Epoch 01/10 — Train: 1.6388 — Val: 1.2950 — Acc: 54.99% ⭐
Epoch 02/10 — Train: 1.2465 — Val: 1.1758 — Acc: 61.25% ⭐
Epoch 03/10 — Train: 1.1447 — Val: 1.1335 — Acc: 59.83%
Epoch 04/10 — Train: 1.0632 — Val: 1.0855 — Acc: 64.39% ⭐
Epoch 05/10 — Train: 1.0714 — Val: 1.1372 — Acc: 60.97%
Epoch 06/10 — Train: 1.0556 — Val: 1.1859 — Acc: 62.11%
Epoch 07/10 — Train: 1.0441 — Val: 1.1172 — Acc: 62.96%
Epoch 08/10 — Train: 1.0456 — Val: 1.1054 — Acc: 63.82%
⏹️  Early stopping epoch 8

✅ Phase A terminée — Meilleure acc : 64.39%


##### Cellule 3 — Phase B : layer4 + FC uniquement (lr ultra petit)

In [35]:
# Dégeler UNIQUEMENT layer4 — pas layer3
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 0.000005},
    {'params': model.fc.parameters(),     'lr': 0.00005},
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=4, factor=0.3
)

EPOCHS_B     = 25
patience_ctr = 0
PATIENCE     = 6

print("=" * 55)
print("PHASE B — layer4 + FC  (lr ultra conservateur)")
print("=" * 55)

for epoch in range(EPOCHS_B):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy finale : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

PHASE B — layer4 + FC  (lr ultra conservateur)
Epoch 01/25 — Train: 0.9287 — Val: 1.0554 — Acc: 65.81% ⭐
Epoch 02/25 — Train: 0.8479 — Val: 1.0308 — Acc: 66.38% ⭐
Epoch 03/25 — Train: 0.7722 — Val: 1.0042 — Acc: 67.81% ⭐
Epoch 04/25 — Train: 0.7062 — Val: 1.0025 — Acc: 66.67%
Epoch 05/25 — Train: 0.6546 — Val: 1.0077 — Acc: 68.95% ⭐
Epoch 06/25 — Train: 0.6216 — Val: 1.0148 — Acc: 68.09%
Epoch 07/25 — Train: 0.5626 — Val: 0.9915 — Acc: 68.38%
Epoch 08/25 — Train: 0.5325 — Val: 1.0060 — Acc: 68.38%
Epoch 09/25 — Train: 0.4885 — Val: 0.9805 — Acc: 69.80% ⭐
Epoch 10/25 — Train: 0.4439 — Val: 0.9947 — Acc: 69.52%
Epoch 11/25 — Train: 0.3996 — Val: 0.9986 — Acc: 69.80%
Epoch 12/25 — Train: 0.3883 — Val: 1.0218 — Acc: 68.95%
Epoch 13/25 — Train: 0.3497 — Val: 1.0163 — Acc: 70.66% ⭐
Epoch 14/25 — Train: 0.3199 — Val: 0.9983 — Acc: 69.80%
Epoch 15/25 — Train: 0.2988 — Val: 1.0248 — Acc: 70.37%
Epoch 16/25 — Train: 0.2773 — Val: 1.0248 — Acc: 70.09%
Epoch 17/25 — Train: 0.2558 — Val: 1.0476 — A

# Nouvelle commence Réentraîner 4 avec ces corrections

In [39]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

print("✅ Imports OK")

✅ Imports OK


In [40]:
BASE       = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'
MODEL_DIR  = '/content/drive/MyDrive/geoscanner/models'
MODEL_PATH = os.path.join(MODEL_DIR, 'resnet50_best.pth')
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Device : {device}")
print(f"✅ BASE   : {BASE}")

✅ Device : cuda
✅ BASE   : /content/drive/MyDrive/geoscanner/dataset/Rock Data


In [41]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("✅ Transformations OK")

✅ Transformations OK


In [42]:
train_ds = datasets.ImageFolder(os.path.join(BASE, 'train'), train_transform)
valid_ds = datasets.ImageFolder(os.path.join(BASE, 'valid'), val_transform)
test_ds  = datasets.ImageFolder(os.path.join(BASE, 'test'),  val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"✅ Train : {len(train_ds)} images")
print(f"✅ Valid : {len(valid_ds)} images")
print(f"✅ Test  : {len(test_ds)} images")
print(f"✅ Classes : {train_ds.classes}")

✅ Train : 3687 images
✅ Valid : 351 images
✅ Test  : 174 images
✅ Classes : ['Basalt', 'Clay', 'Conglomerate', 'Diatomite', 'Shale-(Mudstone)', 'Siliceous-sinter', 'chert', 'gypsum', 'olivine-basalt']


In [43]:
model = models.resnet50(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 9)
)

model     = model.to(device)
criterion = nn.CrossEntropyLoss()

print("✅ Modèle prêt")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 90.0MB/s]


✅ Modèle prêt


### Phase A

In [44]:
model.fc.requires_grad_(True)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

EPOCHS_A     = 10
best_acc     = 0.0
patience_ctr = 0
PATIENCE     = 4
history      = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("PHASE A — FC uniquement  lr=0.001")
print("=" * 55)

for epoch in range(EPOCHS_A):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Phase A terminée — Meilleure acc : {best_acc:.2f}%")

PHASE A — FC uniquement  lr=0.001
Epoch 01/10 — Train: 1.7975 — Val: 1.3838 — Acc: 54.99% ⭐
Epoch 02/10 — Train: 1.4786 — Val: 1.3317 — Acc: 54.42%
Epoch 03/10 — Train: 1.4220 — Val: 1.1928 — Acc: 61.82% ⭐
Epoch 04/10 — Train: 1.4188 — Val: 1.2130 — Acc: 58.12%
Epoch 05/10 — Train: 1.3474 — Val: 1.2653 — Acc: 55.27%
Epoch 06/10 — Train: 1.3239 — Val: 1.2406 — Acc: 58.12%
Epoch 07/10 — Train: 1.3183 — Val: 1.2311 — Acc: 56.41%
⏹️  Early stopping epoch 7

✅ Phase A terminée — Meilleure acc : 61.82%


In [45]:
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 0.000005},
    {'params': model.fc.parameters(),     'lr': 0.00005},
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=4, factor=0.3
)

EPOCHS_B     = 25
patience_ctr = 0
PATIENCE     = 6

print("=" * 55)
print("PHASE B — layer4 + FC  lr ultra conservateur")
print("=" * 55)

for epoch in range(EPOCHS_B):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy finale : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

PHASE B — layer4 + FC  lr ultra conservateur
Epoch 01/25 — Train: 1.2821 — Val: 1.1688 — Acc: 60.40%
Epoch 02/25 — Train: 1.2016 — Val: 1.1353 — Acc: 60.97%
Epoch 03/25 — Train: 1.1501 — Val: 1.1082 — Acc: 62.39% ⭐
Epoch 04/25 — Train: 1.0915 — Val: 1.0995 — Acc: 62.96% ⭐
Epoch 05/25 — Train: 1.0569 — Val: 1.0812 — Acc: 65.24% ⭐
Epoch 06/25 — Train: 1.0330 — Val: 1.0759 — Acc: 64.96%
Epoch 07/25 — Train: 1.0066 — Val: 1.0809 — Acc: 63.82%
Epoch 08/25 — Train: 0.9882 — Val: 1.0867 — Acc: 64.10%
Epoch 09/25 — Train: 0.9517 — Val: 1.0327 — Acc: 66.10% ⭐
Epoch 10/25 — Train: 0.9030 — Val: 1.0183 — Acc: 66.67% ⭐
Epoch 11/25 — Train: 0.8815 — Val: 1.0276 — Acc: 67.24% ⭐
Epoch 12/25 — Train: 0.8712 — Val: 1.0259 — Acc: 67.81% ⭐
Epoch 13/25 — Train: 0.8419 — Val: 1.0284 — Acc: 66.10%
Epoch 14/25 — Train: 0.8023 — Val: 1.0453 — Acc: 66.67%
Epoch 15/25 — Train: 0.7666 — Val: 1.0398 — Acc: 66.38%
Epoch 16/25 — Train: 0.7893 — Val: 1.0517 — Acc: 66.38%
Epoch 17/25 — Train: 0.7375 — Val: 1.0477 — A

### Évaluation du modèle final

In [46]:
model.load_state_dict(torch.load(MODEL_PATH))
model.eval() # Mettre le modèle en mode évaluation

test_loss, correct, total = 0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        test_loss += criterion(outputs, labels).item()
        correct  += outputs.max(1)[1].eq(labels).sum().item()
        total    += labels.size(0)

test_acc = 100. * correct / total
print(f"Accuracy finale sur l'ensemble de test: {test_acc:.2f}%")
print(f"Loss finale sur l'ensemble de test: {test_loss/len(test_loader):.4f}")

Accuracy finale sur l'ensemble de test: 67.82%
Loss finale sur l'ensemble de test: 0.8987
